# Stage C: Counterfactual Decoder Training (Colab)

**UltraBERT-Gen GPT-2 Decoder Training Pipeline**

This notebook trains the 13th head - the Counterfactual Decoder using GPT-2 Medium with prefix injection.

**Why GPT-2 instead of MoE:**
- Pre-trained on 40GB WebText = strong language prior
- Needs less fine-tuning data (MoE needs 8.4B tokens, we have 22M)
- 355M params, ~1GB VRAM

**Prerequisites:**
- Stage B checkpoint (`outputs/modernbert-v2-for-v3-transfer`)
- Counterfactual training data (`data/counterfactual/training`)

**Features:**
- Automatic resume after Colab 18-hour timeout
- Checkpoints saved every 500 steps (~30 min)
- Signal handling for graceful shutdown
- Google Drive persistence

## 1. Environment Setup

In [ ]:
# Check Python and PyTorch versions
import sys
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)

In [ ]:
# Install Flash Attention (pre-built wheel for Colab)
!pip install https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/download/v0.5.4/flash_attn-2.6.3+cu124torch2.9-cp312-cp312-linux_x86_64.whl

In [ ]:
# Verify Flash Attention installation
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"Flash SDP Enabled: {torch.backends.cuda.flash_sdp_enabled()}")

import flash_attn
print(f"Flash Attention Version: {flash_attn.__version__}")

In [ ]:
# Check GPU
!nvidia-smi

# Check if we're on Colab
import os
IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ
print(f"\nRunning on Colab: {IN_COLAB}")

## 2. Repository Setup

In [ ]:
# Clone repository (if on Colab)
import os

REPO_URL = "https://github.com/Pkansagra-hub/Family_osModernBERT.git"
REPO_DIR = "Modeling_studio"

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        print("Cloning repository...")
        !git clone {REPO_URL} {REPO_DIR}
    else:
        print("Repository already exists, pulling latest...")
        !cd {REPO_DIR} && git pull

    os.chdir(REPO_DIR)
    print(f"Working directory: {os.getcwd()}")
else:
    # Local development - assume we're in the repo root
    print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
print("Installing dependencies...")
!pip install -q -e .
!pip install -q wandb tensorboard h5py sacrebleu
!pip install -e . 'datasets>=2.14.0,<3.0.0' -q

# Install familyos_ultrabert wheel (required for embedding generation)
!pip install -q https://github.com/Pkansagra-hub/Family_osModernBERT/releases/download/v2.2.1/familyos_ultrabert-2.2.1-py3-none-any.whl

print("Dependencies installed!")

In [ ]:
# Mount Google Drive for persistent storage
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Create directories on Drive
    DRIVE_BASE = "/content/drive/MyDrive/FamilyOS_ModernBERT"
    !mkdir -p "{DRIVE_BASE}/data"
    !mkdir -p "{DRIVE_BASE}/outputs"
    !mkdir -p "{DRIVE_BASE}/checkpoints"
    !mkdir -p "{DRIVE_BASE}/data/counterfactual"

    # Symlink outputs to Drive for persistence
    !rm -rf outputs checkpoints 2>/dev/null
    !ln -s "{DRIVE_BASE}/outputs" outputs
    !ln -s "{DRIVE_BASE}/checkpoints" checkpoints

    print(f"Data/outputs will be saved to: {DRIVE_BASE}")
else:
    print("Local mode - outputs saved to local directory")

In [ ]:
# Set environment variables
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress TensorFlow warnings
os.environ['TOKENIZERS_PARALLELISM'] = 'false'  # Avoid tokenizer warnings

## 3. Data Verification

In [ ]:
# Verify Stage C prerequisites
import os
from pathlib import Path

def check_stage_c_prerequisites():
    """Verify all prerequisites for Stage C training."""

    print("=" * 60)
    print("STAGE C PREREQUISITES CHECK")
    print("=" * 60)

    # Required: Stage B checkpoint
    stage_b_paths = [
        "outputs/modernbert-v2-for-v3-transfer",
        "outputs/modernbert-v2-for-v3-transfer/checkpoint-18000",
        "checkpoints/modernbert-v2-for-v3-transfer",
    ]

    stage_b_ok = False
    stage_b_path = None
    print("\n[1] Stage B Checkpoint (Required):")
    for path in stage_b_paths:
        if os.path.exists(path):
            # Check for model file
            model_file = os.path.join(path, "pytorch_model.bin")
            safetensors_file = os.path.join(path, "model.safetensors")
            if os.path.exists(model_file) or os.path.exists(safetensors_file):
                stage_b_ok = True
                stage_b_path = path
                print(f"    [OK] Found: {path}")
                break

    if not stage_b_ok:
        print("    [MISSING] Stage B checkpoint not found!")
        print("    Expected at: outputs/modernbert-v2-for-v3-transfer")

    # Check: Synthetic JSONL data (raw input for embedding generation)
    synthetic_paths = [
        "data/counterfactual/synthetic",
        "/content/drive/MyDrive/FamilyOS_ModernBERT/data/counterfactual/synthetic",
    ]

    synthetic_ok = False
    synthetic_path = None
    print("\n[2] Synthetic JSONL Data (for embedding generation):")
    for path in synthetic_paths:
        if os.path.exists(path):
            jsonl_files = [f for f in os.listdir(path) if f.endswith('.jsonl')]
            if jsonl_files:
                synthetic_ok = True
                synthetic_path = path
                total_samples = 0
                for f in jsonl_files:
                    with open(os.path.join(path, f)) as fp:
                        total_samples += sum(1 for _ in fp)
                print(f"    [OK] Found: {path}")
                print(f"         Shards: {len(jsonl_files)}")
                print(f"         Total samples: ~{total_samples:,}")
                break

    if not synthetic_ok:
        print("    [MISSING] No JSONL shards found!")
        print("    Upload to: data/counterfactual/synthetic/")

    # Check: Prepared training data with full sequence embeddings
    training_paths = [
        "data/counterfactual/training",
        "/content/drive/MyDrive/FamilyOS_ModernBERT/data/counterfactual/training",
    ]

    training_ok = False
    training_path = None
    print("\n[3] Prepared Training Data (with embeddings):")
    for path in training_paths:
        if os.path.exists(path):
            samples_file = os.path.join(path, "samples.jsonl")
            seq_embeddings = os.path.join(path, "sequence_embeddings.h5")
            pooled_embeddings = os.path.join(path, "embeddings.h5")

            if os.path.exists(samples_file):
                training_path = path

                # Count samples
                with open(samples_file) as f:
                    num_samples = sum(1 for _ in f)

                has_seq_emb = os.path.exists(seq_embeddings)
                has_pooled_emb = os.path.exists(pooled_embeddings)

                if has_seq_emb:
                    training_ok = True
                    emb_size_gb = os.path.getsize(seq_embeddings) / (1024**3)
                    print(f"    [OK] Found: {path}")
                    print(f"         Samples: {num_samples:,}")
                    print(f"         Full Sequence Embeddings: {emb_size_gb:.2f} GB")
                elif has_pooled_emb:
                    print(f"    [PARTIAL] Found: {path}")
                    print(f"         Samples: {num_samples:,}")
                    print(f"         Pooled Embeddings: Yes")
                    print(f"         Full Sequence: NO - Run section 3.5 to generate!")
                else:
                    print(f"    [PARTIAL] Found samples but no embeddings")
                    print(f"         Run section 3.5 to generate embeddings!")
                break

    if not training_ok and not training_path:
        print("    [NOT READY] Training data not prepared yet")
        print("    Run section 3.5 to generate full sequence embeddings")

    # Summary
    print("\n" + "=" * 60)
    if stage_b_ok and training_ok:
        print("[READY] All prerequisites met! Ready for Stage C training.")
        return True, stage_b_path, training_path
    elif stage_b_ok and synthetic_ok and not training_ok:
        print("[ACTION NEEDED] Run section 3.5 to generate embeddings first!")
        return False, stage_b_path, synthetic_path
    else:
        print("[ERROR] Prerequisites missing. Fix issues above before training.")
        return False, stage_b_path, None
    print("=" * 60)

ready, stage_b_path, data_path = check_stage_c_prerequisites()

## 3.5 Generate Full Sequence Embeddings (GPU)

This step computes encoder embeddings for all counterfactual samples using the A100 GPU.
- **Input**: JSONL files from `data/counterfactual/synthetic/`
- **Output**: HDF5 file with full sequence embeddings (for cross-attention)
- **Time**: ~5-10 min for 100K samples on A100

In [ ]:
%%time
# ============================================================
# GENERATE FULL SEQUENCE EMBEDDINGS ON A100 GPU
# ============================================================
# This converts JSONL counterfactual data → HDF5 embeddings
# Full sequence embeddings enable cross-attention in decoder
#
# NOTE: Output is redirected to log file to prevent Chrome
# from consuming 9GB+ RAM with verbose progress updates!
# ============================================================

import os
from pathlib import Path

# Configuration
SYNTHETIC_DIR = "data/counterfactual/merged"  # Input: JSONL shards
OUTPUT_DIR = "data/counterfactual/training"       # Output: HDF5 + samples
MODEL_PATH = "outputs/modernbert-v2-for-v3-transfer/checkpoint-18000"

# A100-optimized settings
BATCH_SIZE = 128      # Large batch for A100 (reduce to 64 for V100/T4)
MAX_LENGTH = 256      # Max input sequence length
MAX_SAMPLES = ""      # Set to "--max-samples 1000" to limit samples for testing

print("=" * 60)
print("FULL SEQUENCE EMBEDDING GENERATION")
print("=" * 60)

# Check for Drive symlink (Colab)
if os.path.exists("/content/drive/MyDrive/FamilyOS_ModernBERT/data/counterfactual/merged"):
    SYNTHETIC_DIR = "/content/drive/MyDrive/FamilyOS_ModernBERT/data/counterfactual/merged"
    OUTPUT_DIR = "/content/drive/MyDrive/FamilyOS_ModernBERT/data/counterfactual/training"
    print(f"Using Google Drive paths")

# Check if synthetic data exists
if not os.path.exists(SYNTHETIC_DIR):
    print(f"\n[ERROR] Synthetic data not found at {SYNTHETIC_DIR}")
    print("Upload your JSONL shards to Google Drive first!")
    print("\nExpected structure:")
    print("  data/counterfactual/merged/")
    print("    shard_0000.jsonl")
    print("    shard_0001.jsonl")
    print("    ...")
else:
    # Count JSONL files and samples
    jsonl_files = sorted([f for f in os.listdir(SYNTHETIC_DIR) if f.endswith('.jsonl') and f.startswith('shard')])
    total_samples = 0
    for f in jsonl_files:
        with open(os.path.join(SYNTHETIC_DIR, f)) as fp:
            total_samples += sum(1 for line in fp if line.strip())

    print(f"\nInput: {SYNTHETIC_DIR}")
    print(f"  Shards: {len(jsonl_files)}")
    print(f"  Total samples: {total_samples:,}")

    # Check if embeddings already exist
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    embeddings_file = os.path.join(OUTPUT_DIR, "sequence_embeddings.h5")

    if os.path.exists(embeddings_file):
        size_gb = os.path.getsize(embeddings_file) / (1024**3)
        print(f"\n[INFO] Embeddings already exist!")
        print(f"  Path: {embeddings_file}")
        print(f"  Size: {size_gb:.2f} GB")
        print("\nTo regenerate, delete the file first:")
        print(f"  !rm {embeddings_file}")
    else:
        print(f"\nOutput: {OUTPUT_DIR}")
        print(f"  Batch size: {BATCH_SIZE}")
        print(f"  Max length: {MAX_LENGTH}")

        # Estimate time
        est_time = total_samples / 20000  # ~20K samples/min on A100
        print(f"\nEstimated time: ~{est_time:.0f} min")
        print("=" * 60)
        print("\nStarting embedding generation...")
        print("Output redirected to embedding_gen.log to save browser RAM")
        print("Check GPU usage: watch nvidia-smi in another cell")
        print("=" * 60)

        # Run with output redirected to log file (prevents Chrome RAM bloat)
        LOG_FILE = os.path.join(OUTPUT_DIR, "embedding_gen.log")
        !python scripts/agents/prepare_decoder_training_data.py \
            --input-dir "{SYNTHETIC_DIR}" \
            --output-dir "{OUTPUT_DIR}" \
            --model-path "{MODEL_PATH}" \
            --full-sequence \
            --batch-size {BATCH_SIZE} \
            --max-length {MAX_LENGTH} \
            --device cuda \
            {MAX_SAMPLES} > "{LOG_FILE}" 2>&1

        # Show last 20 lines of log
        print("\n--- Last 20 lines of log ---")
        !tail -20 "{LOG_FILE}"

        # Verify result
        if os.path.exists(embeddings_file):
            print("\n" + "=" * 60)
            print("[SUCCESS] Full sequence embeddings generated!")
            print("=" * 60)

            # Show output files
            print(f"\nOutput files in {OUTPUT_DIR}:")
            for f in sorted(os.listdir(OUTPUT_DIR)):
                fpath = os.path.join(OUTPUT_DIR, f)
                if os.path.isfile(fpath):
                    size_mb = os.path.getsize(fpath) / (1024 * 1024)
                    print(f"  {f}: {size_mb:.1f} MB")

            # Verify embeddings
            import h5py
            with h5py.File(embeddings_file, 'r') as hf:
                num_tokens = hf['embeddings'].shape[0]
                hidden_dim = hf['embeddings'].shape[1]
                num_samples = hf.attrs.get('num_samples', 'N/A')
                print(f"\nEmbedding stats:")
                print(f"  Total tokens: {num_tokens:,}")
                print(f"  Hidden dim: {hidden_dim}")
                print(f"  Samples: {num_samples:,}")
        else:
            print(f"\n[ERROR] Embedding generation failed!")
            print(f"Check full log: !cat {LOG_FILE}")

## 4. Stage C Training

**Important Notes:**
- Training saves checkpoints every 500 steps (~30 min)
- If Colab disconnects (18-hour limit), use the "Resume Training" cell below
- `--auto_resume` automatically finds the latest checkpoint

In [ ]:
# Pull latest code before training
!cd /content/Modeling_studio && git pull origin main

In [ ]:
%%time

# ============================================================
# STAGE C: GPT-2 DECODER TRAINING (FRESH START)
# ============================================================
# Use this cell for FIRST training run
# For resuming after disconnect, use the next cell instead

import os

print("=" * 60)
print("STAGE C: GPT-2 COUNTERFACTUAL DECODER TRAINING")
print("=" * 60)
print("  - Decoder: GPT-2 Medium (355M params, pre-trained)")
print("  - Encoder: FROZEN (Stage B checkpoint)")
print("  - Connection: Prefix injection")
print("  - Checkpoints: Every 500 steps")
print("=" * 60)

# Run Stage C training with GPT-2 decoder
!python scripts/train_stage_c.py \
    --config configs/training/multitask/stage_c_gpt2.yaml

# Check if training succeeded
output_dir = "outputs/ultrabert-gen-decoder-gpt2-v1"
if os.path.exists(output_dir):
    files = os.listdir(output_dir)
    has_model = any(f.endswith(('.bin', '.safetensors')) for f in files)
    if has_model:
        print("\n" + "=" * 60)
        print("[SUCCESS] STAGE C COMPLETED!")
        print("=" * 60)
    else:
        print("\n" + "=" * 60)
        print("[INFO] Training in progress - checkpoints saved")
        print("=" * 60)
else:
    print("\n" + "=" * 60)
    print("[ERROR] STAGE C FAILED! Check logs above.")
    print("=" * 60)

### Resume Training (After Colab Disconnect)

**Use this cell if:**
- Colab disconnected due to 18-hour timeout
- You manually stopped training
- Training crashed and you want to continue

The `--auto_resume` flag automatically finds the latest checkpoint.

In [ ]:
%%time

# ============================================================
# RESUME STAGE C TRAINING (AFTER DISCONNECT)
# ============================================================
# Use this cell to RESUME training from latest checkpoint
# --auto_resume automatically finds checkpoint-XXXX folder

import os

print("=" * 60)
print("RESUMING STAGE C GPT-2 DECODER TRAINING")
print("=" * 60)

# Check for existing checkpoints
output_dir = "outputs/ultrabert-gen-decoder-gpt2-v1"
if os.path.exists(output_dir):
    checkpoints = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
    if checkpoints:
        latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
        print(f"  Found {len(checkpoints)} checkpoint(s)")
        print(f"  Latest: {latest}")
    else:
        print("  No checkpoints found - will start fresh")
else:
    print("  No previous training found - will start fresh")

print("=" * 60)

# Resume training with --auto_resume
!python scripts/train_stage_c.py \
    --config configs/training/multitask/stage_c_gpt2.yaml \
    --auto_resume

# Check completion
if os.path.exists(output_dir):
    files = os.listdir(output_dir)
    has_model = any(f.endswith(('.bin', '.safetensors')) for f in files)
    if has_model:
        print("\n" + "=" * 60)
        print("[SUCCESS] STAGE C COMPLETED!")
        print("=" * 60)

## 5. Verify Training Output

In [ ]:
# Verify Stage C output
import os
import json

stage_c_output = "outputs/ultrabert-gen-decoder-gpt2-v1"

if os.path.exists(stage_c_output):
    print(f"Stage C output: {stage_c_output}")
    print("\nFiles:")

    total_size = 0
    for f in sorted(os.listdir(stage_c_output)):
        fpath = os.path.join(stage_c_output, f)
        if os.path.isfile(fpath):
            size = os.path.getsize(fpath) / 1e6
            total_size += size
            print(f"    {f} ({size:.1f} MB)")
        elif os.path.isdir(fpath):
            print(f"    {f}/ (checkpoint)")

    print(f"\nTotal size: {total_size:.1f} MB")

    # Load training state if available
    trainer_state = os.path.join(stage_c_output, "trainer_state.json")
    if os.path.exists(trainer_state):
        with open(trainer_state) as f:
            state = json.load(f)
        print(f"\nTraining Progress:")
        print(f"    Global step: {state.get('global_step', 'N/A')}")
        print(f"    Epoch: {state.get('epoch', 'N/A'):.2f}")

        # Get best metric if available
        if 'best_metric' in state:
            print(f"    Best loss: {state['best_metric']:.4f}")

    # Load eval results
    eval_path = os.path.join(stage_c_output, "eval_results.json")
    if os.path.exists(eval_path):
        with open(eval_path) as f:
            results = json.load(f)
        print("\nEval Results:")
        for k, v in sorted(results.items()):
            if isinstance(v, float):
                print(f"    {k}: {v:.4f}")
else:
    print(f"[INFO] Stage C output not found at {stage_c_output}")
    print("       Training may still be in progress.")

In [ ]:
# Check decoder parameter count
import torch
import os

stage_c_output = "outputs/ultrabert-gen-decoder-gpt2-v1"

def count_parameters(checkpoint_path):
    """Count parameters in the decoder head."""
    try:
        # Try loading the state dict
        for fname in ["pytorch_model.bin", "model.safetensors"]:
            fpath = os.path.join(checkpoint_path, fname)
            if os.path.exists(fpath):
                if fname.endswith(".bin"):
                    state_dict = torch.load(fpath, map_location="cpu")
                else:
                    from safetensors.torch import load_file
                    state_dict = load_file(fpath)

                # Count decoder parameters
                decoder_params = 0
                encoder_params = 0
                head_params = 0

                for name, param in state_dict.items():
                    num_params = param.numel()
                    if "counterfactual" in name.lower() or "decoder" in name.lower() or "gpt2" in name.lower():
                        decoder_params += num_params
                    elif "encoder" in name.lower():
                        encoder_params += num_params
                    else:
                        head_params += num_params

                total = decoder_params + encoder_params + head_params
                print(f"\nParameter Count:")
                print(f"    Encoder (frozen): {encoder_params / 1e6:.1f}M")
                print(f"    Existing heads:   {head_params / 1e6:.1f}M")
                print(f"    GPT-2 Decoder:    {decoder_params / 1e6:.1f}M")
                print(f"    Total:            {total / 1e6:.1f}M")
                return

        print("No model file found in checkpoint")
    except Exception as e:
        print(f"Could not load model: {e}")

if os.path.exists(stage_c_output):
    count_parameters(stage_c_output)

## 6. Training Summary & Backup

In [ ]:
# Training Summary
import os
from datetime import datetime

print("=" * 60)
print("STAGE C TRAINING SUMMARY (GPT-2 Decoder)")
print("=" * 60)
print(f"   Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Check outputs
stage_b_ok = os.path.exists("outputs/modernbert-v2-for-v3-transfer")
stage_c_output = "outputs/ultrabert-gen-decoder-gpt2-v1"
stage_c_ok = os.path.exists(stage_c_output)

# Check for final model
stage_c_complete = False
if stage_c_ok:
    files = os.listdir(stage_c_output)
    stage_c_complete = any(f.endswith(('.bin', '.safetensors')) and not f.startswith('checkpoint') for f in files)

print(f"\n   Stage B (base model): {'[OK]' if stage_b_ok else '[MISSING]'}")
print(f"   Stage C (GPT-2 decoder): {'[COMPLETE]' if stage_c_complete else '[IN PROGRESS]' if stage_c_ok else '[NOT STARTED]'}")

if stage_c_ok:
    # Count checkpoints
    checkpoints = [d for d in os.listdir(stage_c_output) if d.startswith("checkpoint-")]
    print(f"\n   Checkpoints saved: {len(checkpoints)}")
    if checkpoints:
        latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
        step = int(latest.split("-")[1])
        print(f"   Latest checkpoint: {latest} (step {step})")

print("\n" + "=" * 60)
print("OUTPUT LOCATIONS")
print("=" * 60)

if stage_b_ok:
    print("   Stage B: outputs/modernbert-v2-for-v3-transfer")
    print("            -> 12-head encoder (frozen during Stage C)")

if stage_c_ok:
    print("   Stage C: outputs/ultrabert-gen-decoder-gpt2-v1")
    print("            -> 13th head: GPT-2 Counterfactual Decoder")
    print("            -> ~355M parameters (pre-trained GPT-2 Medium)")

print("\n" + "=" * 60)
print("NEXT STEPS")
print("=" * 60)
if stage_c_complete:
    print("   1. Run evaluation metrics (Milestone 15)")
    print("   2. Test generation quality with sample inputs")
    print("   3. Deploy model for inference")
else:
    print("   1. Wait for training to complete OR")
    print("   2. Resume training if disconnected (use Resume cell)")
    print("   3. Run evaluation after training completes")
print("=" * 60)

In [ ]:
# Backup outputs to Google Drive (with timestamp)
if IN_COLAB:
    import shutil
    from datetime import datetime

    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    backup_dir = f"/content/drive/MyDrive/FamilyOS_ModernBERT/runs/stage_c_gpt2_{timestamp}"

    print(f"Creating backup at: {backup_dir}")
    os.makedirs(backup_dir, exist_ok=True)

    # Copy Stage C output
    stage_c_output = "outputs/ultrabert-gen-decoder-gpt2-v1"
    if os.path.exists(stage_c_output):
        shutil.copytree(
            stage_c_output,
            f"{backup_dir}/ultrabert-gen-decoder-gpt2-v1",
            dirs_exist_ok=True
        )
        print("   [OK] Stage C GPT-2 output backed up")

        # Calculate size
        total_size = 0
        for root, dirs, files in os.walk(f"{backup_dir}/ultrabert-gen-decoder-gpt2-v1"):
            for f in files:
                total_size += os.path.getsize(os.path.join(root, f))
        print(f"   Backup size: {total_size / 1e9:.2f} GB")
    else:
        print("   [SKIP] No Stage C output to backup")

    print(f"\nBackup complete: {backup_dir}")
else:
    print("Not on Colab - backup to Drive skipped")

## 7. Test Generation (Optional)

Quick test to verify the decoder generates reasonable output.

In [ ]:
# Quick generation test (run after training completes)
import torch
import os
from transformers import AutoTokenizer

def test_generation():
    """Test the trained GPT-2 decoder with a sample input."""
    try:
        from modeling_studio.models import ModernBertMultiTaskModel
        from modeling_studio.data.labels import Capability

        print("Loading model...")
        model = ModernBertMultiTaskModel.load_checkpoint(
            "outputs/ultrabert-gen-decoder-gpt2-v1",
            device="cuda" if torch.cuda.is_available() else "cpu"
        )
        model.eval()

        tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

        # Test input
        test_input = "I yelled at my kids this morning and now I feel terrible about it."

        print(f"\nInput: {test_input}")
        print("\nGenerating counterfactual...")

        # Encode input
        inputs = tokenizer(test_input, return_tensors="pt")
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        # Get encoder output
        with torch.no_grad():
            encoder_output = model.encoder(**inputs)
            hidden_states = encoder_output.last_hidden_state

            # Generate with decoder head
            if "counterfactual" in model.heads:
                decoder_head = model.heads["counterfactual"]

                # Use full sequence embeddings for prefix injection
                attention_mask = inputs['attention_mask']

                # Generate
                generated_ids = decoder_head.generate(
                    encoder_hidden_states=hidden_states,
                    encoder_attention_mask=attention_mask,
                    max_new_tokens=128,
                    temperature=0.7,
                    top_p=0.9,
                    repetition_penalty=1.2,
                )

                output_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
                print(f"\nCounterfactual: {output_text}")
            else:
                print("Counterfactual head not found in model")
                print(f"Available heads: {list(model.heads.keys())}")

    except Exception as e:
        print(f"Generation test failed: {e}")
        import traceback
        traceback.print_exc()
        print("\nThis is expected if training is not complete.")

# Only run if model exists
if os.path.exists("outputs/ultrabert-gen-decoder-gpt2-v1"):
    test_generation()
else:
    print("Model not found - run training first")

---

# Stage C v2: Continued Fine-tuning (Subdomain Rebalancing)

**Why v2 Fine-tuning?**
- v1 model trained on imbalanced data (some subdomains had <10 samples!)
- v2 uses balanced dataset with ~1000 samples per subdomain
- Lower learning rate (3e-5 vs 1e-4) to prevent catastrophic forgetting
- Only 2-3 epochs (model already learned the task)

**Data Strategy:**
1. Generate synthetic data for underrepresented subdomains
2. Sample from merged folder for well-represented domains (anti-forgetting)
3. Combine into ~86K balanced training samples

**Checkpoints:**
- Input: `outputs/ultrabert-gen-decoder-v1` (already fine-tuned)
- Output: `outputs/ultrabert-gen-decoder-v2` (rebalanced)

## 8. Prepare Balanced Training Data (v2)

In [ ]:
# Check subdomain distribution before preparing balanced data
import os
import json
from pathlib import Path
from collections import defaultdict

def analyze_subdomain_distribution():
    """Analyze subdomain distribution across both data folders."""

    # Paths
    merged_path = Path("data/counterfactual/merged")
    synthetic_path = Path("data/counterfactual/synthetic")

    # Check Drive paths if on Colab
    if IN_COLAB:
        drive_merged = Path("/content/drive/MyDrive/FamilyOS_ModernBERT/data/counterfactual/merged")
        drive_synthetic = Path("/content/drive/MyDrive/FamilyOS_ModernBERT/data/counterfactual/synthetic")
        if drive_merged.exists():
            merged_path = drive_merged
        if drive_synthetic.exists():
            synthetic_path = drive_synthetic

    def count_by_subdomain(folder: Path) -> dict:
        counts = defaultdict(int)
        if not folder.exists():
            return dict(counts)
        for f in folder.glob("*.jsonl"):
            with open(f, "r", encoding="utf-8") as fp:
                for line in fp:
                    try:
                        data = json.loads(line.strip())
                        subdomain = data.get("subdomain", "unknown")
                        counts[subdomain] += 1
                    except:
                        pass
        return dict(counts)

    print("=" * 70)
    print("SUBDOMAIN DISTRIBUTION ANALYSIS")
    print("=" * 70)

    # Load counts
    print(f"\nMerged folder: {merged_path}")
    merged = count_by_subdomain(merged_path)
    merged_total = sum(merged.values())
    print(f"  Total samples: {merged_total:,}")

    print(f"\nSynthetic folder: {synthetic_path}")
    synthetic = count_by_subdomain(synthetic_path)
    synthetic_total = sum(synthetic.values())
    print(f"  Total samples: {synthetic_total:,}")

    # Combine
    combined = defaultdict(int)
    for sub, count in merged.items():
        combined[sub] += count
    for sub, count in synthetic.items():
        combined[sub] += count

    combined_total = merged_total + synthetic_total
    print(f"\nCombined total: {combined_total:,}")

    # Show underrepresented subdomains
    target = 1000  # Target per subdomain
    print(f"\n{'Subdomain':45s} {'Count':>8s} {'Status':>12s}")
    print("-" * 70)

    gaps = []
    for sub, count in sorted(combined.items(), key=lambda x: x[1]):
        if count < target:
            gap = target - count
            gaps.append((sub, count, gap))
            status = f"GAP: {gap}"
        else:
            status = "OK"
        print(f"{sub:45s} {count:8,} {status:>12s}")

    print("-" * 70)
    print(f"Subdomains below target ({target}): {len(gaps)}")
    print(f"Total gap to fill: {sum(g[2] for g in gaps):,}")

    return merged_path, synthetic_path, combined

merged_path, synthetic_path, distribution = analyze_subdomain_distribution()

In [ ]:
%%time
# Prepare balanced training dataset for v2 fine-tuning
# This combines:
# 1. Synthetic data (underrepresented subdomains)
# 2. Sampled merged data (well-represented subdomains - anti-forgetting)

import os
import json
import random
from pathlib import Path
from collections import defaultdict

# Configuration
TARGET_PER_SUBDOMAIN = 1000  # Target samples per subdomain
RANDOM_SEED = 42

# All expected subdomains (86 total)
EXPECTED_SUBDOMAINS = {
    'parenting': ['parenting_toddlers', 'parenting_teens', 'parenting_discipline', 'parenting_education', 'parenting_bonding', 'parenting_siblings', 'parenting_milestones', 'parenting_screen_time'],
    'relationship': ['relationship_spouse', 'relationship_inlaws', 'relationship_grandparents', 'relationship_extended', 'relationship_conflicts', 'relationship_trust', 'relationship_communication', 'relationship_friends'],
    'health': ['health_mental', 'health_children', 'health_elderly', 'health_chronic', 'health_nutrition', 'health_sleep', 'health_exercise', 'health_preventive'],
    'emotions': ['emotions_stress', 'emotions_anger', 'emotions_anxiety', 'emotions_grief', 'emotions_loneliness', 'emotions_overwhelm'],
    'communication': ['communication_arguments', 'communication_listening', 'communication_boundaries', 'communication_difficult_conversations', 'communication_family_meetings'],
    'work': ['work_career', 'work_burnout', 'work_childcare', 'work_remote', 'work_boundaries'],
    'time': ['time_scheduling', 'time_prioritization', 'time_delegation', 'time_quality_time', 'time_procrastination'],
    'routine': ['routine_morning', 'routine_evening', 'routine_meals', 'routine_chores', 'routine_self_care', 'routine_commute'],
    'finance': ['finance_budgeting', 'finance_savings', 'finance_debt', 'finance_education', 'finance_family_expenses'],
    'caregiving': ['caregiving_elderly', 'caregiving_special_needs', 'caregiving_respite', 'caregiving_babysitting', 'caregiving_coordination'],
    'cultural': ['cultural_traditions', 'cultural_festivals', 'cultural_religious', 'cultural_heritage', 'cultural_rituals'],
    'social': ['social_isolation', 'social_community', 'social_friendships', 'social_support_networks', 'social_neighborhood'],
    'home': ['home_organization', 'home_maintenance', 'home_safety', 'home_decoration', 'home_moves'],
    'tech': ['tech_screen_addiction', 'tech_social_media', 'tech_online_safety', 'tech_digital_boundaries', 'tech_family_apps'],
    'life': ['life_weddings', 'life_births', 'life_deaths', 'life_graduations', 'life_relocations'],
}

ALL_SUBDOMAINS = set()
for subs in EXPECTED_SUBDOMAINS.values():
    ALL_SUBDOMAINS.update(subs)

def load_samples_by_subdomain(folder: Path) -> dict:
    """Load all samples grouped by subdomain."""
    samples = defaultdict(list)
    if not folder.exists():
        return dict(samples)
    for f in folder.glob("*.jsonl"):
        with open(f, "r", encoding="utf-8") as fp:
            for line in fp:
                try:
                    data = json.loads(line.strip())
                    subdomain = data.get("subdomain", "unknown")
                    samples[subdomain].append(data)
                except:
                    pass
    return dict(samples)

# Determine paths
merged_path = Path("data/counterfactual/merged")
synthetic_path = Path("data/counterfactual/synthetic")
output_dir = Path("data/counterfactual/training_v2")

if IN_COLAB:
    drive_base = Path("/content/drive/MyDrive/FamilyOS_ModernBERT/data/counterfactual")
    if (drive_base / "merged").exists():
        merged_path = drive_base / "merged"
    if (drive_base / "synthetic").exists():
        synthetic_path = drive_base / "synthetic"
    output_dir = drive_base / "training_v2"

print("=" * 70)
print("PREPARING BALANCED TRAINING DATASET (v2)")
print("=" * 70)
print(f"Merged folder: {merged_path}")
print(f"Synthetic folder: {synthetic_path}")
print(f"Output: {output_dir}")
print(f"Target per subdomain: {TARGET_PER_SUBDOMAIN}")
print()

# Load samples
print("Loading samples...")
merged = load_samples_by_subdomain(merged_path)
synthetic = load_samples_by_subdomain(synthetic_path)

merged_total = sum(len(v) for v in merged.values())
synthetic_total = sum(len(v) for v in synthetic.values())
print(f"  Merged: {merged_total:,} samples")
print(f"  Synthetic: {synthetic_total:,} samples")

# Create output directory
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / "samples.jsonl"

# Sample and write balanced dataset
random.seed(RANDOM_SEED)
total_samples = 0
subdomain_counts = defaultdict(int)

print("\nSampling balanced data...")
print(f"{'Subdomain':45s} {'Merged':>8s} {'Synth':>8s} {'Sampled':>8s}")
print("-" * 70)

with open(output_file, "w", encoding="utf-8") as fp:
    for subdomain in sorted(ALL_SUBDOMAINS):
        merged_samples = merged.get(subdomain, [])
        synthetic_samples = synthetic.get(subdomain, [])

        # Priority: take from synthetic first (new data), then fill from merged
        take_synthetic = min(len(synthetic_samples), TARGET_PER_SUBDOMAIN)
        remaining = TARGET_PER_SUBDOMAIN - take_synthetic
        take_merged = min(len(merged_samples), remaining)

        # Sample
        selected = []
        if take_synthetic > 0:
            selected.extend(random.sample(synthetic_samples, take_synthetic))
        if take_merged > 0:
            selected.extend(random.sample(merged_samples, take_merged))

        # Write
        for sample in selected:
            fp.write(json.dumps(sample, ensure_ascii=False) + "\n")
            total_samples += 1
            subdomain_counts[subdomain] += 1

        total_selected = take_synthetic + take_merged
        status = "OK" if total_selected >= TARGET_PER_SUBDOMAIN else f"GAP: {TARGET_PER_SUBDOMAIN - total_selected}"
        print(f"{subdomain:45s} {len(merged_samples):8,} {len(synthetic_samples):8,} {total_selected:8,}")

print("-" * 70)
print(f"Total samples written: {total_samples:,}")
print(f"Subdomains covered: {len(subdomain_counts)}")
print(f"\nBalanced dataset saved to: {output_file}")

## 8.5 Generate Embeddings for v2 Training Data

Generate full sequence embeddings for the balanced training data using the **v1 checkpoint**.
This enables cross-attention in the decoder during training.

In [ ]:
%%time
# Generate embeddings for v2 training data using the v1 decoder checkpoint
# This uses the SAME encoder as v1 (frozen ModernBERT)

import os
from pathlib import Path

# Configuration
TRAINING_V2_DIR = "data/counterfactual/training_v2"
# Use v1 checkpoint for embedding generation (same encoder)
MODEL_PATH = "outputs/ultrabert-gen-decoder-v1"
BATCH_SIZE = 128
MAX_LENGTH = 256

# Check Drive paths
if IN_COLAB:
    drive_base = "/content/drive/MyDrive/FamilyOS_ModernBERT"
    if os.path.exists(f"{drive_base}/data/counterfactual/training_v2"):
        TRAINING_V2_DIR = f"{drive_base}/data/counterfactual/training_v2"
    if os.path.exists(f"{drive_base}/outputs/ultrabert-gen-decoder-v1"):
        MODEL_PATH = f"{drive_base}/outputs/ultrabert-gen-decoder-v1"

print("=" * 60)
print("GENERATING EMBEDDINGS FOR V2 TRAINING DATA")
print("=" * 60)
print(f"Input: {TRAINING_V2_DIR}")
print(f"Model: {MODEL_PATH}")
print(f"Batch size: {BATCH_SIZE}")

# Check if samples.jsonl exists
samples_file = os.path.join(TRAINING_V2_DIR, "samples.jsonl")
embeddings_file = os.path.join(TRAINING_V2_DIR, "sequence_embeddings.h5")

if not os.path.exists(samples_file):
    print(f"\n[ERROR] samples.jsonl not found at {samples_file}")
    print("Run the previous cell to generate balanced training data first!")
else:
    # Count samples
    with open(samples_file) as f:
        num_samples = sum(1 for _ in f)
    print(f"Samples: {num_samples:,}")

    if os.path.exists(embeddings_file):
        size_gb = os.path.getsize(embeddings_file) / (1024**3)
        print(f"\n[INFO] Embeddings already exist!")
        print(f"  Path: {embeddings_file}")
        print(f"  Size: {size_gb:.2f} GB")
        print("\nTo regenerate, delete the file first:")
        print(f"  !rm {embeddings_file}")
    else:
        # Estimate time
        est_time = num_samples / 20000
        print(f"\nEstimated time: ~{est_time:.0f} min")
        print("=" * 60)

        LOG_FILE = os.path.join(TRAINING_V2_DIR, "embedding_gen_v2.log")

        # Generate embeddings using samples.jsonl directly
        !python scripts/agents/prepare_decoder_training_data.py \
            --input-file "{samples_file}" \
            --output-dir "{TRAINING_V2_DIR}" \
            --model-path "{MODEL_PATH}" \
            --full-sequence \
            --batch-size {BATCH_SIZE} \
            --max-length {MAX_LENGTH} \
            --device cuda > "{LOG_FILE}" 2>&1

        # Show last 20 lines of log
        print("\n--- Last 20 lines of log ---")
        !tail -20 "{LOG_FILE}"

        # Verify result
        if os.path.exists(embeddings_file):
            print("\n" + "=" * 60)
            print("[SUCCESS] V2 embeddings generated!")
            print("=" * 60)

            import h5py
            with h5py.File(embeddings_file, 'r') as hf:
                num_tokens = hf['embeddings'].shape[0]
                hidden_dim = hf['embeddings'].shape[1]
                print(f"  Tokens: {num_tokens:,}")
                print(f"  Hidden dim: {hidden_dim}")
        else:
            print(f"\n[ERROR] Embedding generation failed!")
            print(f"Check log: !cat {LOG_FILE}")

## 9. Stage C v2 Training (Continued Fine-tuning)

**Key Differences from v1:**
- Loads from `ultrabert-gen-decoder-v1` (already fine-tuned)
- Lower learning rate: 3e-5 (vs 1e-4 in v1)
- Fewer epochs: 3 (vs 7 in v1)
- Balanced subdomain data

In [ ]:
# Verify v2 prerequisites before training
import os

print("=" * 60)
print("STAGE C v2 PREREQUISITES CHECK")
print("=" * 60)

# Check v1 checkpoint
v1_path = "outputs/ultrabert-gen-decoder-v1"
if IN_COLAB:
    drive_v1 = "/content/drive/MyDrive/FamilyOS_ModernBERT/outputs/ultrabert-gen-decoder-v1"
    if os.path.exists(drive_v1):
        v1_path = drive_v1

v1_ok = os.path.exists(v1_path)
if v1_ok:
    # Check for model file
    has_model = any(
        os.path.exists(os.path.join(v1_path, f))
        for f in ["pytorch_model.bin", "model.safetensors"]
    )
    v1_ok = has_model

print(f"\n[1] v1 Checkpoint (base for v2):")
if v1_ok:
    print(f"    [OK] {v1_path}")
else:
    print(f"    [MISSING] {v1_path}")
    print("    Train v1 first using cells above!")

# Check v2 training data
v2_data_path = "data/counterfactual/training_v2"
if IN_COLAB:
    drive_v2 = "/content/drive/MyDrive/FamilyOS_ModernBERT/data/counterfactual/training_v2"
    if os.path.exists(drive_v2):
        v2_data_path = drive_v2

samples_ok = os.path.exists(os.path.join(v2_data_path, "samples.jsonl"))
embeddings_ok = os.path.exists(os.path.join(v2_data_path, "sequence_embeddings.h5"))

print(f"\n[2] v2 Training Data:")
if samples_ok and embeddings_ok:
    # Count samples
    with open(os.path.join(v2_data_path, "samples.jsonl")) as f:
        num_samples = sum(1 for _ in f)
    print(f"    [OK] {v2_data_path}")
    print(f"         Samples: {num_samples:,}")
    print(f"         Embeddings: Yes")
elif samples_ok:
    print(f"    [PARTIAL] {v2_data_path}")
    print(f"         Samples: Yes")
    print(f"         Embeddings: MISSING - Run section 8.5!")
else:
    print(f"    [MISSING] {v2_data_path}")
    print("    Run section 8 to prepare balanced data!")

# Summary
print("\n" + "=" * 60)
if v1_ok and samples_ok and embeddings_ok:
    print("[READY] All v2 prerequisites met! Ready for continued training.")
else:
    print("[NOT READY] Fix issues above before training.")
print("=" * 60)

In [ ]:
%%time
# ============================================================
# STAGE C v2: CONTINUED FINE-TUNING (SUBDOMAIN REBALANCING)
# ============================================================
# This trains on balanced subdomain data to fix weak areas
# Loads from ultrabert-gen-decoder-v1 (already fine-tuned)
# Lower LR (3e-5) and fewer epochs (3) to prevent forgetting

import os

print("=" * 60)
print("STAGE C v2: CONTINUED GPT-2 DECODER TRAINING")
print("=" * 60)
print("  - Base: ultrabert-gen-decoder-v1 (already fine-tuned)")
print("  - Data: Balanced ~86K samples (1000/subdomain)")
print("  - LR: 3e-5 (lower to prevent forgetting)")
print("  - Epochs: 3")
print("=" * 60)

# Pull latest code
!cd /content/Modeling_studio && git pull origin main 2>/dev/null || true

# Run Stage C v2 training
!python scripts/train_stage_c.py \
    --config configs/training/multitask/stage_c_gpt2_v2.yaml

# Check if training succeeded
output_dir = "outputs/ultrabert-gen-decoder-v2"
if IN_COLAB:
    drive_out = "/content/drive/MyDrive/FamilyOS_ModernBERT/outputs/ultrabert-gen-decoder-v2"
    if os.path.exists(drive_out):
        output_dir = drive_out

if os.path.exists(output_dir):
    files = os.listdir(output_dir)
    has_model = any(f.endswith(('.bin', '.safetensors')) for f in files)
    if has_model:
        print("\n" + "=" * 60)
        print("[SUCCESS] STAGE C v2 COMPLETED!")
        print("=" * 60)
    else:
        checkpoints = [d for d in files if d.startswith("checkpoint-")]
        if checkpoints:
            latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
            print("\n" + "=" * 60)
            print(f"[IN PROGRESS] Latest checkpoint: {latest}")
            print("=" * 60)
else:
    print("\n" + "=" * 60)
    print("[ERROR] STAGE C v2 FAILED! Check logs above.")
    print("=" * 60)

## 10. Test v2 Generation (Before/After Comparison)

Test the rebalanced model on previously weak subdomains.

In [ ]:
# Compare v1 vs v2 on previously weak subdomains
import torch
import os

# Test scenarios for previously weak subdomains
TEST_SCENARIOS = {
    "health_mental": "I've been feeling really overwhelmed and anxious lately, can't sleep properly and it's affecting my work.",
    "relationship_spouse": "My wife and I keep fighting about money and it's creating tension in our relationship.",
    "relationship_inlaws": "My mother-in-law constantly criticizes my parenting and it makes family gatherings uncomfortable.",
    "emotions_grief": "It's been 6 months since my mother passed away but I still cry every day.",
    "routine_morning": "Every morning is chaos getting the kids ready for school, we're always running late.",
    "routine_commute": "My 2-hour commute is exhausting and I barely have energy left for my family.",
    "parenting_toddlers": "My 2-year-old throws tantrums every time we leave the playground.",
}

def test_model(checkpoint_path: str, model_name: str):
    """Test a model on weak subdomain scenarios."""
    try:
        from scripts.infer_decoder_fp16 import CounterfactualInference

        print(f"\n{'='*60}")
        print(f"Testing: {model_name}")
        print(f"{'='*60}")

        inference = CounterfactualInference(
            checkpoint_path=checkpoint_path,
            device="cuda" if torch.cuda.is_available() else "cpu"
        )

        for subdomain, scenario in TEST_SCENARIOS.items():
            print(f"\n[{subdomain}]")
            print(f"  Input: {scenario[:80]}...")

            result = inference.generate(
                scenario,
                max_new_tokens=128,
                temperature=0.7,
                top_p=0.9
            )

            output = result.get("counterfactual", result.get("output", ""))
            print(f"  Output: {output[:150]}...")

    except Exception as e:
        print(f"Error testing {model_name}: {e}")

# Determine paths
v1_path = "outputs/ultrabert-gen-decoder-v1"
v2_path = "outputs/ultrabert-gen-decoder-v2"

if IN_COLAB:
    drive_base = "/content/drive/MyDrive/FamilyOS_ModernBERT/outputs"
    if os.path.exists(f"{drive_base}/ultrabert-gen-decoder-v1"):
        v1_path = f"{drive_base}/ultrabert-gen-decoder-v1"
    if os.path.exists(f"{drive_base}/ultrabert-gen-decoder-v2"):
        v2_path = f"{drive_base}/ultrabert-gen-decoder-v2"

# Test both models
if os.path.exists(v1_path):
    test_model(v1_path, "v1 (Original)")
else:
    print(f"v1 not found at {v1_path}")

if os.path.exists(v2_path):
    test_model(v2_path, "v2 (Rebalanced)")
else:
    print(f"\nv2 not found at {v2_path}")
    print("Run Stage C v2 training first!")

## 11. v2 Training Summary

In [ ]:
# v2 Training Summary
import os
import json
from datetime import datetime

print("=" * 70)
print("STAGE C v2 TRAINING SUMMARY")
print("=" * 70)
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Check outputs
v1_path = "outputs/ultrabert-gen-decoder-v1"
v2_path = "outputs/ultrabert-gen-decoder-v2"

if IN_COLAB:
    drive_base = "/content/drive/MyDrive/FamilyOS_ModernBERT/outputs"
    if os.path.exists(f"{drive_base}/ultrabert-gen-decoder-v1"):
        v1_path = f"{drive_base}/ultrabert-gen-decoder-v1"
    if os.path.exists(f"{drive_base}/ultrabert-gen-decoder-v2"):
        v2_path = f"{drive_base}/ultrabert-gen-decoder-v2"

v1_ok = os.path.exists(v1_path)
v2_ok = os.path.exists(v2_path)

print(f"\n  v1 (Original):     {'[OK]' if v1_ok else '[MISSING]'} {v1_path}")
print(f"  v2 (Rebalanced):   {'[OK]' if v2_ok else '[MISSING]'} {v2_path}")

# Show v2 training progress
if v2_ok:
    trainer_state = os.path.join(v2_path, "trainer_state.json")
    if os.path.exists(trainer_state):
        with open(trainer_state) as f:
            state = json.load(f)
        print(f"\n  v2 Training Progress:")
        print(f"    Global step: {state.get('global_step', 'N/A')}")
        print(f"    Epoch: {state.get('epoch', 'N/A'):.2f}")
        if 'best_metric' in state:
            print(f"    Best loss: {state['best_metric']:.4f}")

    # Count checkpoints
    checkpoints = [d for d in os.listdir(v2_path) if d.startswith("checkpoint-")]
    if checkpoints:
        latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
        print(f"    Checkpoints: {len(checkpoints)} (latest: {latest})")

print("\n" + "=" * 70)
print("DEPLOYMENT RECOMMENDATION")
print("=" * 70)
if v2_ok:
    print(f"  Use v2 model for production: {v2_path}")
    print("  - Better coverage of weak subdomains (mental health, spouse, etc.)")
    print("  - Same quality on strong domains (anti-forgetting)")
elif v1_ok:
    print(f"  Use v1 model for now: {v1_path}")
    print("  - Run v2 training to improve weak subdomain coverage")
else:
    print("  No trained models available yet!")
print("=" * 70)